
# Exemplo Prático: Soma Paralela no Modelo **SPMD** (Python)

Este notebook implementa, passo a passo, o exemplo mostrado nos slides:

- **Mesmo programa para todos:** todos os processos executam o *mesmo* código (`worker_spmd`).
- **Divisão do vetor:** cada processo calcula a soma de **uma fatia** do vetor, definida pelo seu identificador (`pid`) e pelo número total de processos (`n_procs`).
- **Execução independente:** cada processo trabalha de forma autônoma, sem coordenar o progresso dos demais.
- **Combinação dos resultados:** as somas parciais são reunidas para formar a soma final.

> **Observação didática:** em SPMD, a “diferença” entre processos vem do `pid` (rank).  
> O código é o mesmo; o *trecho de dados* processado é que muda.

---

## Referências (para estudo)
- Pacheco, P. *An Introduction to Parallel Programming*. Morgan Kaufmann, 2011.  
- Grama, A. et al. *Introduction to Parallel Computing*. Addison-Wesley, 2003.



## 1) Imports e utilitários

Vamos usar:
- `multiprocessing` para criar processos (paralelismo por **processos**, contornando o GIL).
- `time.perf_counter()` para medir tempo.

Também criaremos uma função utilitária para visualizar as fatias atribuídas a cada processo.


In [ ]:

import multiprocessing as mp
import time
from typing import List, Tuple


In [ ]:

def calcula_fatia(pid: int, n_procs: int, n: int) -> Tuple[int, int]:
    """Calcula a fatia [ini, fim) do vetor de tamanho n atribuída ao processo pid.

    Divisão em blocos contíguos (block partitioning):
        ini = pid * n // n_procs
        fim = (pid + 1) * n // n_procs
    """
    ini = pid * n // n_procs
    fim = (pid + 1) * n // n_procs
    return ini, fim

def mostra_particionamento(n: int, n_procs: int) -> None:
    """Imprime o particionamento do vetor em blocos para cada pid."""
    print(f"Tamanho do vetor (n) = {n}")
    print(f"Número de processos (n_procs) = {n_procs}\n")
    for pid in range(n_procs):
        ini, fim = calcula_fatia(pid, n_procs, n)
        print(f"pid={pid}: fatia [{ini}, {fim}) (tamanho {fim-ini})")



## 2) Visualizando a divisão do vetor (SPMD)

Antes de paralelizar, vamos entender como as fatias são definidas.  
No SPMD, **cada processo** consegue calcular a sua fatia usando apenas:

- `pid` (seu identificador)
- `n_procs` (quantos processos existem)
- `n` (tamanho do vetor)

Vamos testar com um vetor pequeno para ficar visual.


In [ ]:

mostra_particionamento(n=10, n_procs=4)



## 3) Implementando o **worker** SPMD

A função `worker_spmd` é o “mesmo programa para todos”.  
Ela recebe:

- `pid`: identificador do processo  
- `n_procs`: número total de processos  
- `dados`: o vetor completo (lista)  
- `out`: uma fila (`Queue`) para enviar o resultado parcial ao processo “coletor”

Cada processo:
1. Calcula sua fatia (`ini`, `fim`)
2. Soma apenas `dados[ini:fim]`
3. Envia a soma parcial para `out`


In [ ]:

def worker_spmd(pid: int, n_procs: int, dados: List[int], out: "mp.Queue") -> None:
    """Worker SPMD: mesmo código para todos, dados diferentes por pid."""
    n = len(dados)
    ini, fim = calcula_fatia(pid, n_procs, n)
    parcial = sum(dados[ini:fim])
    out.put((pid, ini, fim, parcial))



## 4) Função que executa a soma paralela SPMD

Por boas práticas (e para funcionar bem em diferentes sistemas), vamos encapsular
a criação dos processos em uma função.

Fluxo:
1. Cria a `Queue` para coletar resultados
2. Inicia `n_procs` processos, todos com `worker_spmd`
3. Aguarda o término (`join`)
4. Lê `n_procs` resultados da fila e agrega

> **Nota:** A agregação é uma etapa comum (ex.: `reduce`) em programas SPMD.


In [ ]:

def soma_spmd(dados: List[int], n_procs: int) -> int:
    """Soma paralela no estilo SPMD (block partitioning) usando multiprocessing."""
    if n_procs < 1:
        raise ValueError("n_procs deve ser >= 1")
    if len(dados) == 0:
        return 0

    out = mp.Queue()
    procs = []

    # "Mesmo programa para todos": todos executam worker_spmd
    for pid in range(n_procs):
        p = mp.Process(target=worker_spmd, args=(pid, n_procs, dados, out))
        p.start()
        procs.append(p)

    # Sincronização: aguarda todos finalizarem
    for p in procs:
        p.join()

    # Combinação dos resultados (reduce por soma)
    resultados = [out.get() for _ in range(n_procs)]
    resultados.sort(key=lambda x: x[0])  # ordena por pid para facilitar a leitura didática

    total = 0
    for pid, ini, fim, parcial in resultados:
        # Para aula: mostrar a fatia e soma parcial
        # (Você pode comentar estes prints em experimentos de performance.)
        print(f"pid={pid} somou dados[{ini}:{fim}] -> parcial={parcial}")
        total += parcial

    return total



## 5) Teste rápido com vetor pequeno (verificando corretude)

Vamos validar que:
- A soma paralela (SPMD) é igual à soma sequencial
- As fatias fazem sentido


In [ ]:

dados_pequenos = list(range(1, 21))  # 1..20
print("Soma sequencial:", sum(dados_pequenos))
print("Soma SPMD:")
total_spmd = soma_spmd(dados_pequenos, n_procs=4)
print("Total SPMD:", total_spmd)



## 6) Experimento didático: tamanho maior e medição de tempo

Agora vamos usar um vetor maior e medir o tempo de execução:
- Sequencial
- Paralelo SPMD (com `n_procs` configurável)

> **Atenção:** em Python, paralelismo com processos tem overhead (criação de processos, serialização de dados, IPC).  
> Em alguns tamanhos, o sequencial pode ser mais rápido. O objetivo aqui é **aprender o modelo** e observar trade-offs.


In [ ]:

def mede_tempo(func, *args, repeticoes: int = 3, **kwargs):
    tempos = []
    resultado = None
    for _ in range(repeticoes):
        t0 = time.perf_counter()
        resultado = func(*args, **kwargs)
        t1 = time.perf_counter()
        tempos.append(t1 - t0)
    return resultado, min(tempos), tempos

# Para evitar prints excessivos no teste grande, usamos uma versão "silenciosa"
def soma_spmd_silenciosa(dados: List[int], n_procs: int) -> int:
    if n_procs < 1:
        raise ValueError("n_procs deve ser >= 1")
    if len(dados) == 0:
        return 0

    out = mp.Queue()
    procs = []
    for pid in range(n_procs):
        p = mp.Process(target=worker_spmd, args=(pid, n_procs, dados, out))
        p.start()
        procs.append(p)

    for p in procs:
        p.join()

    total = 0
    for _ in range(n_procs):
        _, _, _, parcial = out.get()
        total += parcial
    return total


In [ ]:

# Ajuste o tamanho conforme a máquina do laboratório.
# Em sala, 1_000_000 pode ser pesado em alguns ambientes; você pode testar 200_000 ou 500_000.
N = 500_000
dados_grandes = list(range(1, N + 1))

seq_res, seq_t, seq_ts = mede_tempo(sum, dados_grandes, repeticoes=3)
print(f"Sequencial: resultado={seq_res}, melhor_tempo={seq_t:.4f}s, tempos={['%.4f' % t for t in seq_ts]}")

n_procs = 4
par_res, par_t, par_ts = mede_tempo(soma_spmd_silenciosa, dados_grandes, n_procs, repeticoes=3)
print(f"SPMD ({n_procs} procs): resultado={par_res}, melhor_tempo={par_t:.4f}s, tempos={['%.4f' % t for t in par_ts]}")

print("Corretude:", "OK" if seq_res == par_res else "ERRO")



## 7) Explorando diferentes números de processos

Vamos medir o tempo para diferentes valores de `n_procs` e observar:
- Onde há ganho
- Onde o overhead domina

> Em geral, o melhor `n_procs` depende do hardware (núcleos), do tamanho do problema e do custo da computação local.


In [ ]:

testes_procs = [1, 2, 4, 8]
resultados = []

for p in testes_procs:
    res, t_best, _ = mede_tempo(soma_spmd_silenciosa, dados_grandes, p, repeticoes=3)
    resultados.append((p, t_best, res))

print("n_procs | tempo_melhor(s) | resultado")
for p, t_best, res in resultados:
    print(f"{p:6d} | {t_best:14.4f} | {res}")



## 8) Conclusões (o que o aluno deve fixar)

1. **SPMD = mesmo programa, dados diferentes.**  
   Todos os processos executam `worker_spmd`; o `pid` define o intervalo de dados.

2. **Particionamento é essencial.**  
   Aqui usamos **blocos contíguos**, mas existem outras estratégias (cíclica, por tarefa, etc.).

3. **Agregação (reduce) fecha o resultado.**  
   Cada processo produz um parcial; o total vem da combinação.

4. **Overhead existe.**  
   Em Python, o custo de processos e comunicação pode superar o ganho em problemas pequenos.

---

## Exercício para a turma (sugestão)
Implemente um programa SPMD que calcule:

\[ (\sum_{i=1}^{N} i)^2 \]

Requisitos:
- Cada processo calcula a soma parcial da sua fatia.
- O processo “coletor” soma as parciais e **eleva ao quadrado**.
- Compare com a versão sequencial e valide a corretude.
